In [11]:
import re
import numpy as np

corpus_file = "birkbeck.dat"

correct_words = set()
misspelling_map = {}

current_correct_word = None

with open(corpus_file, "r", encoding="utf-8", errors="ignore") as file:

    for line in file:

        line = line.strip()

        if not line:
            continue

        if line.startswith("$"):

            current_correct_word = line[1:].strip().lower()

            correct_words.add(current_correct_word)

            if current_correct_word not in misspelling_map:
                misspelling_map[current_correct_word] = []

        else:

            misspelled_word = line.lower()

            if current_correct_word is not None:

                if misspelled_word not in misspelling_map[current_correct_word]:
                    misspelling_map[current_correct_word].append(
                        misspelled_word
                    )


reverse_map = {}

for correct_word, misspellings in misspelling_map.items():

    for misspelled_word in misspellings:

        if misspelled_word not in reverse_map:
            reverse_map[misspelled_word] = []

        reverse_map[misspelled_word].append(correct_word)


print("=" * 60)
print("SEARCH QUERY SPELLING CORRECTOR")
print("=" * 60)

print("\nBirkbeck Corpus Loaded Successfully")
print("Correct Words:", len(correct_words))
print("Misspellings:", len(reverse_map))


def edit_distance(word1, word2):

    rows = len(word1) + 1
    cols = len(word2) + 1

    matrix = np.zeros((rows, cols), dtype=int)

    for i in range(rows):
        matrix[i][0] = i

    for j in range(cols):
        matrix[0][j] = j

    for i in range(1, rows):

        for j in range(1, cols):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,
                matrix[i][j - 1] + 1,
                matrix[i - 1][j - 1] + cost
            )

    return matrix[-1][-1]


def correct_word(word):

    word = word.lower()

    if word in correct_words:
        return word

    if word in reverse_map:

        candidates = reverse_map[word]

        best_word = candidates[0]
        best_distance = edit_distance(word, best_word)

        for candidate in candidates[1:]:

            distance = edit_distance(word, candidate)

            if distance < best_distance:

                best_distance = distance
                best_word = candidate

        return best_word

    best_word = word
    best_distance = float("inf")

    for candidate in correct_words:

        if abs(len(candidate) - len(word)) > 3:
            continue

        distance = edit_distance(word, candidate)

        if distance < best_distance:

            best_distance = distance
            best_word = candidate

    return best_word


def correct_query(query):

    words = re.findall(r"[a-zA-Z_]+", query.lower())

    corrected_words = []
    corrections = []

    for word in words:

        corrected_word = correct_word(word)

        corrected_words.append(corrected_word)

        if word != corrected_word:

            corrections.append(
                (word, corrected_word)
            )

    corrected_query = " ".join(corrected_words)

    return corrected_query, corrections


queries = [
    "Amercia",
    "Apirl",
    "Austrain",
    "Cambrige",
    "Ameracan",
    "bechuarnia_land"
]


for query in queries:

    corrected_query, corrections = correct_query(query)

    print("\n" + "-" * 60)

    print("Original Query:")
    print(query)

    print("\nCorrections:")

    if corrections:

        for wrong, correct in corrections:

            print(wrong, "->", correct)

    else:

        print("No spelling errors found")

    print("\nCorrected Query:")
    print(corrected_query)


print("\n" + "=" * 60)
print("SPELLING CORRECTION COMPLETED")
print("=" * 60)

SEARCH QUERY SPELLING CORRECTOR

Birkbeck Corpus Loaded Successfully
Correct Words: 6130
Misspellings: 33968

------------------------------------------------------------
Original Query:
Amercia

Corrections:
amercia -> america

Corrected Query:
america

------------------------------------------------------------
Original Query:
Apirl

Corrections:
apirl -> april

Corrected Query:
april

------------------------------------------------------------
Original Query:
Austrain

Corrections:
austrain -> austrian

Corrected Query:
austrian

------------------------------------------------------------
Original Query:
Cambrige

Corrections:
cambrige -> cambridge

Corrected Query:
cambridge

------------------------------------------------------------
Original Query:
Ameracan

Corrections:
ameracan -> american

Corrected Query:
american

------------------------------------------------------------
Original Query:
bechuarnia_land

Corrections:
bechuarnia_land -> bechuanaland

Corrected Query:
be